# Lab 3 – Transformata Hougha na liściach ziemniaka

Zbiór i klasy ustawia się w `leaf_utils.py` (`DATA_DIR`, `CLASSES`).
Druga klasa z `CLASSES` jest traktowana jako pozytywna (`y = 1`).

Obserwacje w sekcji 3 dotyczą Task 1: **zdrowe** (`Potato___healthy`) vs **chore** (`Potato___Early_blight`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import color, feature, measure
from skimage.transform import hough_line, hough_line_peaks

import leaf_utils as lu

CLS = list(lu.CLASSES)  # CLS[0] -> y = 0, CLS[1] -> y = 1
# moc wygładzania konturu maski liścia: promień dysku [px] dla dylatacji, a potem erozji (0 = bez wygładzania)
SMOOTH_RADIUS = 5
print("Zbiór:", lu.DATA_DIR.name, "| klasy:", CLS)

## 1. Ładowanie przykładów

In [ ]:
N_PER_CLASS = 4
samples = [(label, path, lu.load(path)) for label, path in lu.sample_paths(N_PER_CLASS)]
for label in lu.CLASSES:
    print(f"{label}: {sum(1 for l, _ in lu.list_paths() if l == label)} obrazów")
print("Rozmiar obrazu:", samples[0][2].shape)

In [ ]:
fig, axes = plt.subplots(len(lu.CLASSES), N_PER_CLASS, figsize=(3 * N_PER_CLASS, 3 * len(lu.CLASSES)))
for ax, (label, path, img) in zip(axes.flat, samples):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Transformata Hougha (cały obraz)

Kroki dla każdego obrazu:
1. konwersja do skali szarości,
2. detekcja krawędzi (Canny),
3. transformata Hougha dla linii – akumulator w przestrzeni (θ, ρ),
4. wybór najsilniejszych pików i narysowanie odpowiadających im prostych na obrazie.

In [ ]:
CANNY_SIGMA = 2.0
N_LINES = 10


def draw_line(ax, angle, dist, shape, **kw):
    # prosta: x*cos(θ) + y*sin(θ) = ρ
    rows, cols = shape[:2]
    x0, y0 = dist * np.cos(angle), dist * np.sin(angle)
    t = np.array([-2, 2]) * max(rows, cols)
    ax.plot(x0 - t * np.sin(angle), y0 + t * np.cos(angle), **kw)
    ax.set_xlim(0, cols)
    ax.set_ylim(rows, 0)


def show_accumulator(ax, h, theta, d, angles, dists):
    extent = [np.rad2deg(theta[0]), np.rad2deg(theta[-1]), d[-1], d[0]]
    ax.imshow(np.log1p(h), extent=extent, cmap="magma", aspect="auto")
    ax.plot(np.rad2deg(angles), dists, "c+", markersize=10)
    ax.set_xlabel("θ [°]")
    ax.set_ylabel("ρ [px]")


for label, _, img in samples:
    edges = feature.canny(color.rgb2gray(img), sigma=CANNY_SIGMA)
    h, theta, d, (_, angles, dists) = lu.hough(edges, num_peaks=N_LINES)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img); axes[0].set_title(f"Oryginał ({label})")
    axes[1].imshow(edges, cmap="gray"); axes[1].set_title("Krawędzie (Canny)")
    show_accumulator(axes[2], h, theta, d, angles, dists); axes[2].set_title("Akumulator Hougha (log)")
    axes[3].imshow(img)
    for angle, dist in zip(angles, dists):
        draw_line(axes[3], angle, dist, img.shape, color="r", linewidth=1)
    axes[3].set_title(f"Wykryte linie (top {N_LINES})")
    for ax in axes[[0, 1, 3]]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Próba ilościowego ujęcia obserwacji

Zauważone na przykładach z Task 1:
1. **krawędzie chorych liści są „brzydsze”** (poszarpane, dużo drobnych kawałków),
2. **piki akumulatora**: u zdrowych dwie grupy kątów, u chorych jedna (+ kilka rozrzuconych),
3. **linie u chorych biegną głównie wzdłuż nerwu głównego**, a u zdrowych także wzdłuż nerwów bocznych.

Żeby kontur liścia i tło nie zagłuszały nerwów, krawędzie liczymy **tylko wewnątrz zerodowanej maski liścia**
(segmentacja ExG z `binaryzacja.ipynb`, kontur wygładzony dylatacją i erozją o promieniu `SMOOTH_RADIUS`).

In [ ]:
prepared = []
for label, _, img in samples:
    leaf = lu.segment_leaf(img, smooth_radius=SMOOTH_RADIUS)["leaf"]
    prepared.append((label, img, leaf, lu.inner_edges(img, leaf, sigma=CANNY_SIGMA)))

fig, axes = plt.subplots(2, len(samples), figsize=(2.5 * len(samples), 5.5))
for col, (label, img, leaf, edges) in enumerate(prepared):
    axes[0, col].imshow(feature.canny(color.rgb2gray(img), sigma=CANNY_SIGMA), cmap="gray")
    axes[0, col].set_title(f"{label}: cały obraz")
    axes[1, col].imshow(edges, cmap="gray")
    axes[1, col].set_title("tylko wnętrze liścia")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.1 „Brzydkie” krawędzie → fragmentacja

Metoda: etykietowanie składowych spójnych mapy krawędzi (`skimage.measure.label`).
Ładny nerw to jedna długa składowa, a plamy i nekrozy dają wiele krótkich kawałków.

Cechy:
- `edge_density` – piksele krawędzi / powierzchnia liścia,
- `edge_fragments` – liczba składowych na 100 pikseli krawędzi (więcej = bardziej poszarpane),
- `edge_mean_len` – średnia długość składowej w pikselach.

In [ ]:
fig, axes = plt.subplots(1, len(prepared), figsize=(2.5 * len(prepared), 3))
for ax, (label, img, leaf, edges) in zip(axes, prepared):
    f = lu.edge_features(edges, leaf)
    lbl = measure.label(edges, connectivity=2)
    ax.imshow(color.label2rgb(lbl, bg_label=0))
    ax.set_title(f"{label}\nfragm={f['edge_fragments']:.2f}, len={f['edge_mean_len']:.0f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Składowe krawędzi (każdy kolor = osobny fragment)")
plt.tight_layout()
plt.show()

### 3.2 Grupy pików w akumulatorze

Metoda: kąty θ wybranych pików traktujemy jako kierunki (okres 180°).
- `peak_clusters` – liczba grup: sortujemy kąty na okręgu i liczymy przerwy większe niż 20°,
- `peak_spread` – rozrzut kierunkowy 1 − |mean(e^{2iθ})| (0 = wszystkie linie równoległe, 1 = rozłożone równomiernie).
  Podwojenie kąta sprawia, że θ i θ+180° to ten sam kierunek.

Liczymy obie wersje: z krawędzi całego obrazu (jak w punkcie 2, razem z konturem liścia) i tylko z wnętrza liścia.

In [ ]:
fig, axes = plt.subplots(2, len(prepared), figsize=(2.5 * len(prepared), 5.5), subplot_kw={"projection": "polar"})
for col, (label, img, leaf, edges) in enumerate(prepared):
    full = feature.canny(color.rgb2gray(img), sigma=CANNY_SIGMA)
    for row, (name, e) in enumerate([("cały obraz", full), ("wnętrze", edges)]):
        h, theta, d, (acc, angles, dists) = lu.hough(e, num_peaks=N_LINES)
        ax = axes[row, col]
        # kierunek jest osiowy -> rysujemy θ i θ+π
        for a, w in zip(angles, acc / acc.max()):
            ax.plot([a, a], [0, w], "r-")
            ax.plot([a + np.pi, a + np.pi], [0, w], "r-")
        ax.set_yticklabels([])
        ax.set_title(f"{label} ({name})\ngrupy={lu.angle_clusters(angles)}, "
                     f"rozrzut={lu.circular_spread(angles):.2f}", fontsize=8)
plt.tight_layout()
plt.show()

### 3.3 Linie wzdłuż nerwu głównego vs nerwy boczne

Metoda:
1. **nerw główny** to najsilniejsza prosta Hougha we wnętrzu liścia (`lu.midrib`).
   Oś elipsy dopasowanej do maski (`regionprops.orientation`) testowałem, ale na okrągłych liściach wskazuje zły kierunek,
2. dla pozostałych prostych liczymy kąt względem nerwu głównego; > 20° traktujemy jako „skośną” (nerw boczny),
3. dodatkowo **probabilistyczny Hough** (`probabilistic_hough_line`) zwraca odcinki zamiast nieskończonych prostych,
   co lepiej pasuje do krótkich nerwów bocznych.

Kolory: niebieska przerywana = nerw główny, zielone = wzdłuż osi, czerwone = skośne.

In [ ]:
fig, axes = plt.subplots(2, len(prepared), figsize=(2.8 * len(prepared), 6))
for col, (label, img, leaf, edges) in enumerate(prepared):
    axis_theta, axis_rho = lu.midrib(edges)

    # pozostałe proste Hougha (pierwsza to nerw główny)
    _, _, _, (_, angles, dists) = lu.hough(edges, num_peaks=N_LINES)
    angles, dists = angles[1:], dists[1:]
    oblique = lu.axial_diff(angles, axis_theta) > np.deg2rad(20)
    ax = axes[0, col]
    ax.imshow(img)
    for a, dd, o in zip(angles, dists, oblique):
        draw_line(ax, a, dd, img.shape, color="r" if o else "lime", linewidth=1)
    draw_line(ax, axis_theta, axis_rho, img.shape, color="b", linestyle="--", linewidth=2)
    ax.set_title(f"{label}: proste\nskośne {oblique.mean():.0%}", fontsize=9)

    # odcinki z probabilistycznego Hougha
    segs, is_obl = lu.vein_segments(edges, axis_theta)
    ax = axes[1, col]
    ax.imshow(img)
    for ((x0, y0), (x1, y1)), o in zip(segs, is_obl):
        ax.plot([x0, x1], [y0, y1], color="r" if o else "lime", linewidth=1.5)
    frac = is_obl.mean() if len(segs) else float("nan")
    ax.set_title(f"odcinki: {len(segs)}, skośne {frac:.0%}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Czy to działa na całym zbiorze?

Na 4 przykładach łatwo zobaczyć wzorzec, którego nie ma. Liczymy wszystkie cechy dla wszystkich obrazów zbioru
i dla każdej sprawdzamy:
- rozkłady w obu klasach (wykresy pudełkowe),
- trafność najlepszego pojedynczego progu (optymistyczna, bo próg dobrany na tych samych danych).

`lesion_frac` (udział plam z maski ExG, zob. `binaryzacja.ipynb`) jest dodany jako punkt odniesienia.
Liczenie trwa około 20 s na Task 1 (305 obrazów) i około 2 min na Task 2 (2000 obrazów).

In [ ]:
rows, y = [], []
for label, path in lu.list_paths():
    img = lu.load(path)
    seg = lu.segment_leaf(img, smooth_radius=SMOOTH_RADIUS)
    f = lu.hough_features(img, seg["leaf"], num_peaks=N_LINES)
    f["lesion_frac"] = seg["lesions"].sum() / seg["leaf"].sum()
    rows.append(f)
    y.append(label == CLS[1])

y = np.array(y)
names = list(rows[0])
X = np.array([[r[n] for n in names] for r in rows], dtype=float)


def best_threshold_acc(x, y):
    ok = ~np.isnan(x)
    x, y = x[ok], y[ok]
    return max(max(np.mean((x > t) == y), np.mean((x <= t) == y)) for t in np.unique(x))


print(f"{'cecha':24s} {'med. ' + CLS[0]:>16s} {'med. ' + CLS[1]:>16s} {'trafność progu':>15s}")
for i, n in enumerate(names):
    x = X[:, i]
    print(f"{n:24s} {np.nanmedian(x[~y]):16.3f} {np.nanmedian(x[y]):16.3f} {best_threshold_acc(x, y):15.1%}")

In [ ]:
ncols = 4
nrows = int(np.ceil(len(names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
for ax, (i, n) in zip(axes.flat, enumerate(names)):
    x = X[:, i]
    ax.boxplot([x[~y & ~np.isnan(x)], x[y & ~np.isnan(x)]])
    ax.set_xticks([1, 2], CLS)
    ax.set_title(f"{n}\n(próg: {best_threshold_acc(x, y):.0%})", fontsize=9)
for ax in axes.flat[len(names):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Wnioski dla Task 1 (zdrowe vs chore)

- Obserwacja 1 (brzydkie krawędzie) **potwierdza się**: chore liście mają wyraźnie więcej i krótszych fragmentów krawędzi.
- Obserwacje 2 i 3 (grupy pików, skośne linie) na całym zbiorze są **słabe** (ok. 55–65%): rozkłady klas mocno się nakładają.
  Najlepiej z nich wypada liczba skośnych odcinków (`segments_oblique_n`) – u zdrowych widać więcej nerwów bocznych,
  ale to częściowo efekt tego, że plamy przerywają nerwy (mniej odcinków w ogóle, `segments_n`).
- Rozrzut pików jest lepszy z konturem liścia (`peak_spread_full`) niż bez niego, więc „dwie grupy” u zdrowych
  pochodzą w dużej mierze z obrysu liścia, a nie z nerwów.
- Najsilniejszą cechą jest **udział plam** z segmentacji kolorem, a nie geometria linii.

## 5. Pokrywanie się i istotność cech

Żeby wybór cech nie „podglądał” wyniku końcowego, najpierw odkładamy **25% danych jako zbiór testowy**.
Wszystkie analizy z tej sekcji i dobór cech w sekcji 6 liczymy **tylko na zbiorze treningowym**.

Brakujące wartości (np. brak odcinków → `segments_oblique_frac` = NaN) uzupełniamy medianą z treningu.

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
imputer = SimpleImputer(strategy="median").fit(X_train)
X_train, X_test = imputer.transform(X_train), imputer.transform(X_test)
names = np.array(names)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def make_rf():
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)


print(f"trening: {len(y_train)} ({y_train.mean():.0%} klasy '{CLS[1]}'), test: {len(y_test)}")

### 5.1 Jak bardzo cechy pokrywają się ze sobą (redundancja)

Korelacja rang Spearmana między cechami. |ρ| bliskie 1 oznacza, że dwie cechy niosą praktycznie tę samą informację
(np. `edge_fragments` i `edge_mean_len` to niemal odwrotność siebie), więc w klasyfikatorze wystarczy jedna z nich.

In [ ]:
rho = np.nan_to_num(spearmanr(X_train).statistic)  # stała kolumna -> NaN -> 0

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(rho, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{rho[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="ρ Spearmana")
ax.set_title("Korelacja między cechami (trening)")
plt.tight_layout()
plt.show()

print("Pary silnie skorelowane (|ρ| > 0.8):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if abs(rho[i, j]) > 0.8:
            print(f"  {names[i]:24s} ~ {names[j]:24s} ρ = {rho[i, j]:+.2f}")

### 5.2 Jak bardzo klasy pokrywają się w każdej cesze (istotność pojedynczych cech)

Dla każdej cechy osobno:
- **OVL** (overlap) – wspólne pole pod histogramami obu klas: 1 = rozkłady identyczne, 0 = całkowicie rozdzielone,
- **AUC** – prawdopodobieństwo, że losowy obraz klasy 1 ma większą wartość cechy niż losowy obraz klasy 0.
  Siła rozdzielania to |AUC − 0.5| · 2 (0 = brak, 1 = idealne),
- **p-value testu Manna–Whitneya** – czy rozkłady cechy w obu klasach istotnie się różnią
  (przy wielu cechach naraz stosujemy poprawkę Bonferroniego: p · liczba cech),
- **informacja wzajemna (MI)** – ile informacji o klasie niesie cecha, wychwytuje też zależności niemonotoniczne.

In [ ]:
def overlap(x0, x1, bins=30):
    edges = np.histogram_bin_edges(np.concatenate([x0, x1]), bins=bins)
    p0 = np.histogram(x0, edges)[0] / len(x0)
    p1 = np.histogram(x1, edges)[0] / len(x1)
    return np.minimum(p0, p1).sum()


mi = mutual_info_classif(X_train, y_train, random_state=SEED)
stats = []
for i, n in enumerate(names):
    x0, x1 = X_train[~y_train, i], X_train[y_train, i]
    auc = roc_auc_score(y_train, X_train[:, i])
    p = mannwhitneyu(x0, x1).pvalue
    stats.append((n, overlap(x0, x1), auc, abs(auc - 0.5) * 2, min(p * len(names), 1.0), mi[i]))
stats.sort(key=lambda s: -s[3])

print(f"{'cecha':24s} {'OVL':>6s} {'AUC':>6s} {'siła':>6s} {'p (Bonf.)':>10s} {'MI':>6s}")
for n, ovl, auc, sep, p, m in stats:
    star = "*" if p < 0.05 else " "
    print(f"{n:24s} {ovl:6.2f} {auc:6.2f} {sep:6.2f} {p:10.1e}{star} {m:6.3f}")
print("* istotne na poziomie 0.05 po poprawce Bonferroniego")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
order = [s[0] for s in stats][::-1]
axes[0].barh(order, [s[1] for s in stats][::-1], color="gray")
axes[0].set_xlabel("OVL – pokrycie klas (mniej = lepiej)")
axes[1].barh(order, [s[3] for s in stats][::-1], color="tab:blue")
axes[1].set_xlabel("|AUC − 0.5| · 2 – siła rozdzielania (więcej = lepiej)")
plt.tight_layout()
plt.show()

### 5.3 Ważność cech w lesie losowym

Miary z 5.2 oceniają każdą cechę osobno. Las losowy ocenia je razem, z uwzględnieniem interakcji:
- **MDI** (mean decrease in impurity) – wbudowana miara drzewa: o ile cecha średnio zmniejsza nieczystość Giniego.
  Szybka, ale faworyzuje cechy ciągłe o wielu unikalnych wartościach,
- **ważność permutacyjna** – o ile spada trafność na danych walidacyjnych, gdy losowo przemieszamy wartości jednej cechy.
  Liczona w 5-krotnej walidacji krzyżowej na treningu.

Uwaga: przy dwóch silnie skorelowanych cechach permutacja jednej mało szkodzi, bo druga ją zastępuje.
Obie mogą wtedy wyglądać na mało ważne, dlatego w sekcji 6 odrzucamy duplikaty z 5.1.

In [ ]:
mdi = make_rf().fit(X_train, y_train).feature_importances_

perm = np.zeros((cv.get_n_splits(), len(names)))
for k, (tr, va) in enumerate(cv.split(X_train, y_train)):
    rf = make_rf().fit(X_train[tr], y_train[tr])
    perm[k] = permutation_importance(rf, X_train[va], y_train[va], n_repeats=10,
                                     random_state=SEED, n_jobs=-1).importances_mean

order = np.argsort(perm.mean(0))
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
axes[0].barh(names[order], mdi[order], color="gray")
axes[0].set_xlabel("MDI")
axes[1].barh(names[order], perm.mean(0)[order], xerr=perm.std(0)[order], color="tab:green")
axes[1].axvline(0, color="k", linewidth=0.8)
axes[1].set_xlabel("ważność permutacyjna (spadek trafności, średnia ± odch. std z 5 foldów)")
plt.tight_layout()
plt.show()

## 6. Klasyfikacja lasem losowym na najważniejszych cechach

1. **Ranking** – cechy sortujemy malejąco po ważności permutacyjnej z 5.3 (remisy rozstrzyga MDI).
2. **Usuwanie duplikatów** – idąc od góry rankingu, pomijamy cechę, jeśli |ρ| z którąś już wybraną przekracza `CORR_MAX`.
3. **Wybór k** – trafność w 5-krotnej walidacji krzyżowej na treningu dla top-1, top-2, …; bierzemy najlepsze k.
4. **Ocena końcowa** na odłożonym zbiorze testowym, w porównaniu z lasem na wszystkich cechach
   i lasem na samych cechach geometrycznych (bez `lesion_frac`, żeby zobaczyć, ile daje sam Hough).

In [ ]:
CORR_MAX = 0.8

ranking = np.lexsort((-mdi, -perm.mean(0)))  # główny klucz: ważność permutacyjna
selected = []
for i in ranking:
    duplicate_of = [j for j in selected if abs(rho[i, j]) > CORR_MAX]
    if duplicate_of:
        print(f"pomijam {names[i]:24s} (|ρ| > {CORR_MAX} z {names[duplicate_of[0]]})")
    else:
        selected.append(i)
print("Ranking po usunięciu duplikatów:", names[selected].tolist())

ks = np.arange(1, len(selected) + 1)
cv_scores = np.array([cross_val_score(make_rf(), X_train[:, selected[:k]], y_train, cv=cv) for k in ks])
best_k = ks[np.argmax(cv_scores.mean(1))]

plt.figure(figsize=(8, 4))
plt.errorbar(ks, cv_scores.mean(1), yerr=cv_scores.std(1), marker="o", capsize=3)
plt.axvline(best_k, color="r", linestyle="--", label=f"najlepsze k = {best_k}")
plt.xticks(ks)
plt.xlabel("liczba cech (top-k z rankingu)")
plt.ylabel("trafność CV (trening)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Wybrane cechy:", names[selected[:best_k]].tolist())

In [ ]:
variants = {
    f"top-{best_k}": selected[:best_k],
    "wszystkie cechy": list(range(len(names))),
    "bez lesion_frac (sam Hough/krawędzie)": [i for i, n in enumerate(names) if n != "lesion_frac"],
}

print(f"{'wariant':40s} {'CV trening':>12s} {'test':>8s}")
models = {}
for name, idx in variants.items():
    cv_acc = cross_val_score(make_rf(), X_train[:, idx], y_train, cv=cv)
    models[name] = make_rf().fit(X_train[:, idx], y_train)
    test_acc = accuracy_score(y_test, models[name].predict(X_test[:, idx]))
    print(f"{name:40s} {cv_acc.mean():6.1%} ±{cv_acc.std():4.1%} {test_acc:8.1%}")

best_name = f"top-{best_k}"
idx = variants[best_name]
pred = models[best_name].predict(X_test[:, idx])
print(f"\nRaport dla '{best_name}' na zbiorze testowym:")
print(classification_report(y_test, pred, target_names=CLS))

fig, axes = plt.subplots(1, len(variants), figsize=(5 * len(variants), 4))
for ax, (name, idx) in zip(axes, variants.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, models[name].predict(X_test[:, idx]),
                                            display_labels=CLS, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()